# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** search visibility / CTR opportunity. March 1–21 is the decision snapshot; no later data or label-derived field enters the rule.


## 1. My rule and its reason codes

**Rule:** review a page when it has at least 500 GSC impressions, ranks 4–20, and weighted CTR is under 0.5%. Prioritize more visible pages and pages nearer the top. **Reason code:** `visible_low_ctr_position_opportunity`. **Action:** `review_title_snippet_and_intent`. This is decision support, not a causal claim.


In [1]:
from pathlib import Path
import os, duckdb, pandas as pd, numpy as np
p=list((Path.home()/'.cache'/'huggingface'/'hub').rglob('fact_content_daily_performance/month=2026-03/data_0.parquet'))

if p: path=str(p[0])
else:
    try:
        from google.colab import userdata; 
        token=userdata.get('HF_TOKEN')
    except Exception: 
        token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if not token: 
        raise RuntimeError('Set HF_TOKEN as a Colab Secret or environment variable; never paste it here.')
    from huggingface_hub import hf_hub_download
    path=hf_hub_download('FlyRank/internship-warehouse','fact_content_daily_performance/month=2026-03/data_0.parquet',repo_type='dataset',token=token)

con=duckdb.connect(); 
safe=path.replace("'","''"); 

con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{safe}')")
sql="""SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) impressions_21d,SUM(gsc_clicks) clicks_21d,SUM(gsc_sum_position)/NULLIF(SUM(gsc_impressions),0) avg_position_21d FROM march WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21' AND gsc_data_available IS TRUE GROUP BY 1,2 HAVING SUM(gsc_impressions)>0"""
d=con.sql(sql).df(); 

d['weighted_ctr_pct']=100*d.clicks_21d/d.impressions_21d; 
print('Decision snapshot items:',len(d))

d['position_bucket']=pd.cut(d.avg_position_21d,[0,3,5,10,20,1e9],labels=['1-3','4-5','6-10','11-20','21+'])
s1=d.groupby('position_bucket',observed=True).agg(n=('content_hash_id','size'),impressions=('impressions_21d','sum'),clicks=('clicks_21d','sum')).assign(weighted_ctr_pct=lambda x:100*x.clicks/x.impressions).reset_index()
print('Signal 1, CTR vs position (FlyRank CTR-fix flag): CONFIRMED — CTR falls down the results; n shown.'); 
display(s1)

d['volume_bucket']=pd.cut(d.impressions_21d,[0,99,499,1999,9999,float('inf')],labels=['1-99','100-499','500-1,999','2,000-9,999','10,000+'])
s2=d.groupby('volume_bucket',observed=True).agg(n=('content_hash_id','size'),median_clicks_21d=('clicks_21d','median'),total_clicks=('clicks_21d','sum')).reset_index()
print('Signal 2, volume (quick-win priority): CONFIRMED — visibility buckets contain more observed clicks; priority not causal lift.'); 
display(s2)


Decision snapshot items: 162301
Signal 1, CTR vs position (FlyRank CTR-fix flag): CONFIRMED — CTR falls down the results; n shown.


,position_bucket,n,impressions,clicks,weighted_ctr_pct
0,1-3,16821,27380357.0,116203.0,0.424403
1,4-5,24997,47140183.0,172840.0,0.366651
2,6-10,50660,44903175.0,135129.0,0.300934
3,11-20,27411,19410193.0,62790.0,0.323490
4,21+,41159,40336775.0,56310.0,0.139600


Signal 2, volume (quick-win priority): CONFIRMED — visibility buckets contain more observed clicks; priority not causal lift.


,volume_bucket,n,median_clicks_21d,total_clicks
0,1-99,73389,0.0,6204.0
1,100-499,38460,0.0,24939.0
2,"500-1,999",29692,2.0,97304.0
3,"2,000-9,999",17573,8.0,229629.0
4,"10,000+",3187,32.0,185249.0


## 2. Build the ranked queue (writes the CSV)

Score = `100 × log1p(impressions) + 20 × (21 − position) − 100 × CTR percentage`, after the transparent eligibility gates.


In [2]:
ok=(d.impressions_21d>=500)&d.avg_position_21d.between(4,20)&(d.weighted_ctr_pct<.5)
q=d.loc[ok,['client_hash_id','content_hash_id','impressions_21d','clicks_21d','avg_position_21d','weighted_ctr_pct']].copy(); 

q['baseline_action_score']=100*np.log1p(q.impressions_21d)+20*(21-q.avg_position_21d)-100*q.weighted_ctr_pct
q['reason_code']='visible_low_ctr_position_opportunity'; 
q['action_label']='review_title_snippet_and_intent'; 

q=q.sort_values('baseline_action_score',ascending=False).reset_index(drop=True); 
q['baseline_rank']=q.index+1

out=Path.cwd()/'work'/'outputs'; 
out.mkdir(parents=True,exist_ok=True); 

q.to_csv(out/'baseline_action_score.csv',index=False); 
print('Wrote work/outputs/baseline_action_score.csv:',len(q),'rows'); 
display(q.head(10))


Wrote work/outputs/baseline_action_score.csv: 23190 rows


,client_hash_id,content_hash_id,impressions_21d,clicks_21d,avg_position_21d,weighted_ctr_pct,baseline_action_score,reason_code,action_label,baseline_rank
0,client_62f4a7e64f5e0096,content_b99ea6861864dea5,125082.0,245.0,4.448818,0.195872,1485.109777,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,1
1,client_73cda7b4e4f265ea,content_471d9cabce329a66,110663.0,275.0,4.544301,0.248502,1465.689142,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,2
2,client_62f4a7e64f5e0096,content_7c6373141eae744a,100270.0,57.0,5.813135,0.056847,1449.615838,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,3
3,client_73cda7b4e4f265ea,content_e578ac84778da489,74466.0,108.0,4.321234,0.145033,1440.883195,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,4
4,client_73cda7b4e4f265ea,content_f43118e089ecc69a,80299.0,129.0,4.853248,0.160650,1436.222563,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,5
5,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,78330.0,155.0,4.581667,0.197881,1435.448449,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,6
6,client_73cda7b4e4f265ea,content_8e1334d6356668e3,63486.0,1.0,4.847573,0.001575,1428.750075,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,7
7,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,60075.0,22.0,4.696130,0.036621,1422.751885,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,8
8,client_62f4a7e64f5e0096,content_132dcd40faff1dc2,59100.0,78.0,4.228562,0.131980,1420.931108,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,9
9,client_e547b89c05043229,content_77276ad7a26f4905,55291.0,121.0,4.033007,0.218842,1409.493995,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,10


## 3. Top-10 review

Each row is a human review of title/snippet/search intent. It is selected because it is visible, ranks 4–20, and has CTR below 0.5%. It would be wrong if branded/navigational queries, a SERP feature, intentional protection, or non-addressable query intent explains low CTR.


In [3]:
top=q.head(10).copy(); 
top['review']='Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/snippet work—explains CTR.'

for r in top.itertuples(): print(f'{r.baseline_rank}. {r.review}')
display(top[['baseline_rank','action_label','reason_code','impressions_21d','avg_position_21d','weighted_ctr_pct','review']])


1. Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/snippet work—explains CTR.
2. Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/snippet work—explains CTR.
3. Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/snippet work—explains CTR.
4. Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/snippet work—explains CTR.
5. Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/snippet work—explains CTR.
6. Action: review title/snippet/intent. Why: visible position-4–20 page with CTR <0.5%. Wrong if query, brand, SERP, or intent context—not page/s

,baseline_rank,action_label,reason_code,impressions_21d,avg_position_21d,weighted_ctr_pct,review
0,1,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,125082.0,4.448818,0.195872,Action: review title/snippet/intent. Why: visi...
1,2,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,110663.0,4.544301,0.248502,Action: review title/snippet/intent. Why: visi...
2,3,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,100270.0,5.813135,0.056847,Action: review title/snippet/intent. Why: visi...
3,4,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,74466.0,4.321234,0.145033,Action: review title/snippet/intent. Why: visi...
4,5,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,80299.0,4.853248,0.160650,Action: review title/snippet/intent. Why: visi...
5,6,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,78330.0,4.581667,0.197881,Action: review title/snippet/intent. Why: visi...
6,7,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,63486.0,4.847573,0.001575,Action: review title/snippet/intent. Why: visi...
7,8,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,60075.0,4.696130,0.036621,Action: review title/snippet/intent. Why: visi...
8,9,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,59100.0,4.228562,0.131980,Action: review title/snippet/intent. Why: visi...
9,10,review_title_snippet_and_intent,visible_low_ctr_position_opportunity,55291.0,4.033007,0.218842,Action: review title/snippet/intent. Why: visi...


## 4. Weak picks + leakage check

Candidates near 500 impressions, position 20, or 0.5% CTR are weak: small measurement changes can change eligibility. I exclude product flags, hashes as inputs, GA4 fields, and all post–March-21 data.


In [4]:
weak=q[(q.impressions_21d<650)|(q.avg_position_21d>18)|(q.weighted_ctr_pct>.4)].tail(10); 
print('Borderline picks:',len(weak)); 
display(weak)

input_columns=['impressions_21d','clicks_21d','avg_position_21d','weighted_ctr_pct']; 
forbidden=['next','future','label','trend','health','priority','flag']; 

assert not any(any(x in c.lower() for x in forbidden) for c in input_columns); 
print('Leakage check passed: no future, label-derived, or product-flag input.')


Borderline picks: 10


,client_hash_id,content_hash_id,impressions_21d,clicks_21d,avg_position_21d,weighted_ctr_pct,baseline_action_score,reason_code,action_label,baseline_rank
23180,client_23a62021009f63c4,content_dc4ce3bbcdc0e49f,563.0,2.0,19.397869,0.355240,630.024075,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23181
23181,client_23a62021009f63c4,content_4b7a77e5566967eb,502.0,2.0,18.669323,0.398406,628.831925,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23182
23182,client_23a62021009f63c4,content_87d78780db7d3c53,517.0,1.0,19.847195,0.193424,628.711257,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23183
23183,client_23a62021009f63c4,content_d65ebc13a7e205d7,506.0,2.0,18.782609,0.395257,627.673235,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23184
23184,client_e547b89c05043229,content_c6171be8a3f6cd2f,604.0,2.0,19.991722,0.331126,627.575826,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23185
23185,client_23a62021009f63c4,content_04489e3eb5d19b23,500.0,1.0,19.928000,0.200000,623.100610,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23186
23186,client_ff644d8251367cbb,content_dae484e29d3b8192,547.0,2.0,19.553931,0.365631,622.985847,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23187
23187,client_23a62021009f63c4,content_e1f66c9dea66b9f0,569.0,2.0,19.834798,0.351494,622.718293,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23188
23188,client_73cda7b4e4f265ea,content_9c16bb26a852f0e8,551.0,2.0,19.840290,0.362976,618.251356,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23189
23189,client_2094c6eb080311d5,content_e78b7a9a34851991,510.0,2.0,19.905882,0.392157,606.303626,visible_low_ctr_position_opportunity,review_title_snippet_and_intent,23190


Leakage check passed: no future, label-derived, or product-flag input.


## 5. Self-check

- Two bucket tables with n; CTR/position is FlyRank flag-linked.
- Score, reason code, action label, and CSV are produced.
- Top ten reviewed; no future-window or label-derived input.
